In [123]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler,LabelEncoder
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats



## LAB-2

### 1.LOAD DATA AND INSPECT

In [100]:
df = pd.read_csv("apartments_for_rent_classified_10K.csv", 
                  encoding='cp1252', 
                  on_bad_lines='skip',
                  engine='python',
                  sep=';')

In [101]:
display (df.head())
print(df.shape)
print(df.info())


,id,category,title,body,amenities,bathrooms,bedrooms,currency,fee,has_photo,...,price_display,price_type,square_feet,address,cityname,state,latitude,longitude,source,time
0,5668626895,housing/rent/apartment,"Studio apartment 2nd St NE, Uhland Terrace NE,...","This unit is located at second St NE, Uhland T...",NaN,NaN,0.0,USD,No,Thumbnail,...,$790,Monthly,101,NaN,Washington,DC,38.9057,-76.9861,RentLingo,1577359415
1,5664597177,housing/rent/apartment,Studio apartment 814 Schutte Road,"This unit is located at 814 Schutte Road, Evan...",NaN,NaN,1.0,USD,No,Thumbnail,...,$425,Monthly,106,814 Schutte Rd,Evansville,IN,37.9680,-87.6621,RentLingo,1577017063
2,5668626833,housing/rent/apartment,"Studio apartment N Scott St, 14th St N, Arling...","This unit is located at N Scott St, 14th St N,...",NaN,1.0,0.0,USD,No,Thumbnail,...,"$1,390",Monthly,107,NaN,Arlington,VA,38.8910,-77.0816,RentLingo,1577359410
3,5659918074,housing/rent/apartment,Studio apartment 1717 12th Ave,"This unit is located at 1717 12th Ave, Seattle...",NaN,1.0,0.0,USD,No,Thumbnail,...,$925,Monthly,116,1717 12th Avenue,Seattle,WA,47.6160,-122.3275,RentLingo,1576667743
4,5668626759,housing/rent/apartment,"Studio apartment Washington Blvd, N Cleveland ...","This unit is located at Washington Blvd, N Cle...",NaN,NaN,0.0,USD,No,Thumbnail,...,$880,Monthly,125,NaN,Arlington,VA,38.8738,-77.1055,RentLingo,1577359401


(10000, 22)
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             10000 non-null  int64  
 1   category       10000 non-null  str    
 2   title          10000 non-null  str    
 3   body           10000 non-null  str    
 4   amenities      6451 non-null   str    
 5   bathrooms      9966 non-null   float64
 6   bedrooms       9993 non-null   float64
 7   currency       10000 non-null  str    
 8   fee            10000 non-null  str    
 9   has_photo      10000 non-null  str    
 10  pets_allowed   5837 non-null   str    
 11  price          10000 non-null  int64  
 12  price_display  10000 non-null  str    
 13  price_type     10000 non-null  str    
 14  square_feet    10000 non-null  int64  
 15  address        6673 non-null   str    
 16  cityname       9923 non-null   str    
 17  state          9923 non-null   str    
 18  latitu

### 2.HANDLE MISSING VALUES

In [102]:
df['amenities'] = df['amenities'].fillna('None')
df['address'] = df['address'].fillna ('Unknown')
df['pets_allowed'] = df['pets_allowed'].fillna('Unknown')
df = df.dropna(subset=['cityname', 'state'])
df['bathrooms'] =df['bathrooms'].fillna(df['bathrooms'].median())
df['bedrooms'] =df['bedrooms'].fillna(df['bedrooms'].median())

print (df.shape)
print (df.isnull().sum())





(9923, 22)
id               0
category         0
title            0
body             0
amenities        0
bathrooms        0
bedrooms         0
currency         0
fee              0
has_photo        0
pets_allowed     0
price            0
price_display    0
price_type       0
square_feet      0
address          0
cityname         0
state            0
latitude         0
longitude        0
source           0
time             0
dtype: int64


### Data Cleaning

- **Categorical columns** (`amenities`, `address`, `pets_allowed`): missing values filled with placeholder labels ("None" / "Unknown") instead of dropping, since these columns had a large amount of missing data.
- **Location columns** (`cityname`, `state`, `latitude`, `longitude`): rows with missing values dropped, since these are important for location-based analysis and only a small number of rows were affected.
- **Numeric columns** (`bathrooms`, `bedrooms`): missing values filled with the **median**, since only a few values were missing.
- **Result:** dataset now has no missing values.

### 3.Encode Categorical Data

In [103]:
print (df.dtypes)

id                 int64
category             str
title                str
body                 str
amenities            str
bathrooms        float64
bedrooms         float64
currency             str
fee                  str
has_photo            str
pets_allowed         str
price              int64
price_display        str
price_type           str
square_feet        int64
address              str
cityname             str
state                str
latitude         float64
longitude        float64
source               str
time               int64
dtype: object


In [106]:
print(df['fee'].unique())
print(df['has_photo'].unique())
print(df['pets_allowed'].unique())
print(df['price_type'].unique())
print(df['category'].unique())


<StringArray>
['No']
Length: 1, dtype: str
<StringArray>
['Thumbnail', 'Yes', 'No']
Length: 3, dtype: str
<StringArray>
['Unknown', 'Cats,Dogs', 'Cats', 'Dogs']
Length: 4, dtype: str
<StringArray>
['Monthly', 'Weekly', 'Monthly|Weekly']
Length: 3, dtype: str
<StringArray>
['housing/rent/apartment', 'housing/rent/home', 'housing/rent/short_term']
Length: 3, dtype: str


In [107]:
cat_cols = df.select_dtypes(include='object').columns
df[cat_cols].nunique()

C:\Users\wasia\AppData\Local\Temp\ipykernel_6812\3925099768.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns


category            3
title            9274
body             9884
amenities        2248
currency            1
fee                 1
has_photo           3
pets_allowed        4
price_display    1725
price_type          3
address          6593
cityname         1574
state              51
source             12
dtype: int64

In [121]:
df = df.drop(columns=['currency', 'fee', 'title', 'body', 'amenities', 'price_display', 'address','latitude','longitude','time'], errors='ignore')

In [109]:

print (df.columns.tolist())
print (df.nunique())
print (df.info())
display(df.head())


['id', 'category', 'bathrooms', 'bedrooms', 'has_photo', 'pets_allowed', 'price', 'price_type', 'square_feet', 'cityname', 'state', 'source']
id              9923
category           3
bathrooms         14
bedrooms          10
has_photo          3
pets_allowed       4
price           1724
price_type         3
square_feet     1734
cityname        1574
state             51
source            12
dtype: int64
<class 'pandas.DataFrame'>
Index: 9923 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            9923 non-null   int64  
 1   category      9923 non-null   str    
 2   bathrooms     9923 non-null   float64
 3   bedrooms      9923 non-null   float64
 4   has_photo     9923 non-null   str    
 5   pets_allowed  9923 non-null   str    
 6   price         9923 non-null   int64  
 7   price_type    9923 non-null   str    
 8   square_feet   9923 non-null   int64  
 9   cityname      9923 non-n

,id,category,bathrooms,bedrooms,has_photo,pets_allowed,price,price_type,square_feet,cityname,state,source
0,5668626895,housing/rent/apartment,1.0,0.0,Thumbnail,Unknown,790,Monthly,101,Washington,DC,RentLingo
1,5664597177,housing/rent/apartment,1.0,1.0,Thumbnail,Unknown,425,Monthly,106,Evansville,IN,RentLingo
2,5668626833,housing/rent/apartment,1.0,0.0,Thumbnail,Unknown,1390,Monthly,107,Arlington,VA,RentLingo
3,5659918074,housing/rent/apartment,1.0,0.0,Thumbnail,Unknown,925,Monthly,116,Seattle,WA,RentLingo
4,5668626759,housing/rent/apartment,1.0,0.0,Thumbnail,Unknown,880,Monthly,125,Arlington,VA,RentLingo


In [111]:
df = pd.get_dummies(df, columns=['category', 'has_photo', 'pets_allowed', 'price_type'], drop_first=True)

print(df.shape)
display(df.head())

(9923, 17)


,id,bathrooms,bedrooms,price,square_feet,cityname,state,source,category_housing/rent/home,category_housing/rent/short_term,has_photo_Thumbnail,has_photo_Yes,"pets_allowed_Cats,Dogs",pets_allowed_Dogs,pets_allowed_Unknown,price_type_Monthly|Weekly,price_type_Weekly
0,5668626895,1.0,0.0,790,101,Washington,DC,RentLingo,False,False,True,False,False,False,True,False,False
1,5664597177,1.0,1.0,425,106,Evansville,IN,RentLingo,False,False,True,False,False,False,True,False,False
2,5668626833,1.0,0.0,1390,107,Arlington,VA,RentLingo,False,False,True,False,False,False,True,False,False
3,5659918074,1.0,0.0,925,116,Seattle,WA,RentLingo,False,False,True,False,False,False,True,False,False
4,5668626759,1.0,0.0,880,125,Arlington,VA,RentLingo,False,False,True,False,False,False,True,False,False


### explanation pending  

### 4. Normalize and Scale Numeric Columns

In [120]:
num_cols = ['bathrooms', 'bedrooms', 'price', 'square_feet']

scaler_std = StandardScaler()
df_std = df.copy()
df_std[num_cols] = scaler_std.fit_transform(df_std[num_cols])

# Normalization: range [0, 1]

scaler_minmax = MinMaxScaler()
df_norm = df.copy()
df_norm[num_cols] = scaler_minmax.fit_transform(df_norm[num_cols])

print("Summary statistics after scaling:")
print("Standardized Data Stats:")

display(df_std[num_cols].describe().round(3))

print("\nNormalized Data Stats:")
display(df_norm[num_cols].describe().round(3))

display(df.head().round(3))

Summary statistics after scaling:
Standardized Data Stats:


,bathrooms,bedrooms,price,square_feet
count,9923.000,9923.000,9923.000,9923.000
mean,0.000,0.000,0.000,0.000
std,1.000,1.000,1.000,1.000
min,-0.616,-1.851,-1.193,-1.286
25%,-0.616,-0.790,-0.498,-0.452
50%,-0.616,0.271,-0.200,-0.218
75%,1.008,0.271,0.194,0.234
max,11.568,7.699,47.320,59.426



Normalized Data Stats:


,bathrooms,bedrooms,price,square_feet
count,9923.000,9923.000,9923.000,9923.000
mean,0.051,0.194,0.025,0.021
std,0.082,0.105,0.021,0.016
min,0.000,0.000,0.000,0.000
25%,0.000,0.111,0.014,0.014
50%,0.000,0.222,0.020,0.018
75%,0.133,0.222,0.029,0.025
max,1.000,1.000,1.000,1.000


,id,bathrooms,bedrooms,price,square_feet,cityname,state,source,category_housing/rent/home,category_housing/rent/short_term,has_photo_Thumbnail,has_photo_Yes,"pets_allowed_Cats,Dogs",pets_allowed_Dogs,pets_allowed_Unknown,price_type_Monthly|Weekly,price_type_Weekly
0,5668626895,-0.616,-1.851,-0.646,-1.286,Washington,DC,RentLingo,False,False,True,False,False,False,True,False,False
1,5664597177,-0.616,-0.790,-0.984,-1.278,Evansville,IN,RentLingo,False,False,True,False,False,False,True,False,False
2,5668626833,-0.616,-1.851,-0.089,-1.277,Arlington,VA,RentLingo,False,False,True,False,False,False,True,False,False
3,5659918074,-0.616,-1.851,-0.521,-1.263,Seattle,WA,RentLingo,False,False,True,False,False,False,True,False,False
4,5668626759,-0.616,-1.851,-0.562,-1.250,Arlington,VA,RentLingo,False,False,True,False,False,False,True,False,False


### Feature Scaling (Standardization vs. Normalization)

We experimented with two primary scaling techniques on our numeric columns (`bathrooms`, `bedrooms`, `price`, `square_feet`):
1. **Standardization (`StandardScaler`)**: Centers the data around a mean of 0 with a standard deviation of 1.
2. **Normalization (`MinMaxScaler`)**: Compresses all data values strictly between the range of 0 and 1.

#### Why Standardization is the Better Choice for This Dataset:
* **The Outlier Squash Problem:** Our dataset contains extreme luxury apartments (massive outliers in `price` and `square_feet`). Under Normalization, forcing these huge maximums to `1.0` squashes 75% of our regular data points into a tiny fraction between `0.0` and `0.01`, making them indistinguishable.
* **Outlier Resilience:** Standardization does not have strict boundaries. It allows regular apartments to stay naturally clustered around `0` while letting the extreme luxury properties safely shoot up to high values (like `47.3` and `59.4`) without ruining the distribution of the rest of the data.


### Mathematical Formulas for Feature Scaling

#### 1. StandardScaler (Standardization)
Standardization shifts the data to have a mean of 0 and a standard deviation of 1 using the following formula:

$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

Where:
* $x$ = Original value
* $\mu$ = Mean (average) of the column
* $\sigma$ = Standard deviation of the column

---

#### 2. MinMaxScaler (Normalization)
Normalization rescales the data so that all values fit strictly within a specified range (by default, 0 to 1) using the following formula:

$$x_{\text{scaled}} = \frac{x - x_{\text{min}}}{x_{\text{max}} - x_{\text{min}}}$$

Where:
* $x$ = Original value
* $x_{\text{min}}$ = Minimum value in the column
* $x_{\text{max}}$ = Maximum value in the column



### 5. Remove Duplicates

In [119]:

before = df_std.shape[0]

df_clean = df_std.drop_duplicates()

after = df_clean.shape[0]

print(f"Removed {before - after} duplicate rows.")
print(f"New dataset shape: {df_clean.shape}")

Removed 0 duplicate rows.
New dataset shape: (9923, 17)


## Lab 3: Exploratory Data Analysis (EDA)

In [124]:
print("Shape:", df_clean.shape)
display(df_clean.head())

Shape: (9923, 17)


,id,bathrooms,bedrooms,price,square_feet,cityname,state,source,category_housing/rent/home,category_housing/rent/short_term,has_photo_Thumbnail,has_photo_Yes,"pets_allowed_Cats,Dogs",pets_allowed_Dogs,pets_allowed_Unknown,price_type_Monthly|Weekly,price_type_Weekly
0,5668626895,-0.616476,-1.850832,-0.645731,-1.286022,Washington,DC,RentLingo,False,False,True,False,False,False,True,False,False
1,5664597177,-0.616476,-0.789715,-0.984303,-1.278414,Evansville,IN,RentLingo,False,False,True,False,False,False,True,False,False
2,5668626833,-0.616476,-1.850832,-0.089175,-1.276893,Arlington,VA,RentLingo,False,False,True,False,False,False,True,False,False
3,5659918074,-0.616476,-1.850832,-0.520506,-1.263198,Seattle,WA,RentLingo,False,False,True,False,False,False,True,False,False
4,5668626759,-0.616476,-1.850832,-0.562248,-1.249503,Arlington,VA,RentLingo,False,False,True,False,False,False,True,False,False
